## 1) 랜덤 포레스트
* 배깅 : 같은 알고리즘으로 여러 개의 분류기를 만들어 보팅으로 최종 결정하는 알고리즘
* 배깅의 대표적 알고리즘이 랜덤 포레스트
* 앙상블 알고리즘 중 비교적 수행 속도가 빠름
* 다양한 영역에서 높은 예측성능을 보임
* 결정 트리가 기반 알고리즘 -> 쉽고 직관적임
* 여러 개의 결정트리 분류기가 전체 데이터에서 배깅 방식으로 각자의 데이터를 샘플링해 학습을 수행한 뒤, 최종적으로 모든 분류기가 보팅을 통해 예측 결정하게 됨
* 부트스트랩(bootstrapping) 분할방식 : 여러 개의 데이터셋을 중첩되게 분리하는 것
  * 개별적 분류기 기반 알고리즘은 결정 트리
  * 개별 트리가 학습하는 데이터셋은 전체 데이터에서 일부가 중첩되게 샘플링된 데이터셋 -> 부트스트랩
  * 배깅bagging = bootstrap aggregating의 준말
  * 통계학 : 여러 개의 작은 데이터셋을 임의로 만들어 개별 평균의 분포도를 측정하는 등의 목적을 위한 샘플링 방식
  * 랜덤포레스트의 subset(서브세트) 데이터는 부트스트래핑으로 데이터가 임의로 만들어짐
* 사이킷런 RandomForestClassifier

In [7]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd


### feature 중복 제거하는 함수
def get_new_feature_name_df(old_feature_name_df):
    feature_dup_df = pd.DataFrame(data=old_feature_name_df.groupby('column_name').cumcount(), columns=['dup_cnt'])
    feature_dup_df = feature_dup_df.reset_index()
    new_feature_name_df = pd.merge(old_feature_name_df.reset_index(), feature_dup_df, how='outer')
    new_feature_name_df['column_name'] = new_feature_name_df[['column_name', 'dup_cnt']].apply(lambda x: str(x[0]) + '_' + str(x[1]) if x[1]>0 else x[0], axis=1)
    new_feature_name_df = new_feature_name_df.drop(['index'], axis=1)
    return new_feature_name_df

### train 폴더의 학습용피처/레이블 데이터셋, test 폴더의 테스트피처 데이터/레이블 파일 각각 로드하는 함수
def get_human_dataset():
    # 각 데이터파일은 공백으로 분리됨 -> sep을 공백으로 할당
    feature_name_df = pd.read_csv(r'../human_activity/features.txt', sep='\+', header=None, names=['column_index', 'column_name'])
    # 중복된 피처명 수정하는 함수 이용
    new_feature_name_df = get_new_feature_name_df(feature_name_df)
    # DataFrame에 피처명을 칼럼으로 부여하기 위해 리스트 객체로 다시 변환
    feature_name = new_feature_name_df.iloc[:, 1].values.tolist()
    # 학습 피처 데이터셋과 테스트 피처 데이터셋을 dataFrame으로 로딩
    x_train = pd.read_csv(r'../human_activity/train/X_train.txt', sep='\s+', names=feature_name)
    x_test = pd.read_csv(r'../human_activity/test/X_test.txt', sep='\s+', names=feature_name)
    #학습 레이블과 테스트 레이블 데이터를 DataFrame으로 로딩. 칼럼명은 action 으로 부여
    y_train = pd.read_csv(r'../human_activity/train/y_train.txt', sep='\s+', header=None, names=['action'])
    y_test = pd.read_csv(r'../human_activity/test/y_test.txt', sep='\s+', header=None, names=['action'])
    #로드된 학습/테스트용 DataFrame을 모두 반환
    return x_train, x_test, y_train, y_test

In [8]:
### 사용자 행동 인식 예측 데이터셋 사용
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# 결정트리에서 사용한 get_human_datset() 이용해 학습, 테스트용 DataFrame 반환
x_train, x_test, y_train, y_test = get_human_dataset()

# 랜덤 포레스트 학습 및 별도의 테스트셋으로 예측 성능 평가
rf_clf = RandomForestClassifier(random_state=0)
rf_clf.fit(x_train, y_train)
pred = rf_clf.predict(x_test)
accuracy = accuracy_score(y_test, pred)
print('랜덤 포레스트 정확도 : {0:.4f}'.format(accuracy))

랜덤 포레스트 정확도 : 0.9253


## 2) 랜덤 포레스트 하이퍼 파라미터 튜닝
* 트리 기반의 앙상블 알고리즘 단점 : 하이퍼 파라미터가 너무 많고, 튜닝 시간이 많이 소모됨
* 트리 기반 자체 하이퍼 파라미터 많음 + 배깅, 부스팅, 학습, 정규화 등을 위한 파라미터 추가됨
* 그나마 랜덤 포레스트가 적은 편 & 결정트리에서 사용하는 하이퍼 파라미터와 같은 것이 대부분
* n_estimators(결정 트리 개수. 디폴트=10), max_features(디폴트=auto(sqrt와 같음)), max_depth, min_samples_leaf

In [10]:
### GridSearchCV로 랜덤 포레스트의 하이퍼 파라미터 튜닝

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

params = {'n_estimators':[100], 'max_depth':[6, 8, 10, 12], 'min_samples_leaf':[8, 12, 18], 'min_samples_split':[8, 16, 20]}

rf_clf = RandomForestClassifier(random_state=0, n_jobs=-1)
grid_cv = GridSearchCV(rf_clf, param_grid=params, cv=2, n_jobs=-1)
grid_cv.fit(x_train, y_train)

print('최적 하이퍼 파라미너 : \n', grid_cv.best_params_)
print('최고 예측 정확도 : {0:.4f}'.format(grid_cv.best_score_))

KeyboardInterrupt: 

In [ ]:
### 하이퍼 파라미터 적용 후 별도 테스트 셋에서 예측 성능 측정

rf_clf1 = RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=8, min_samples_split=8, random_state=0)
rf_clf1.fit(x_train, y_train)
pred - rf_clf1.predict(x_test)

print('최고 예측 정확도 : {0:.4f}'.format(accuracy_score(y_test, pred)))

In [ ]:
### 피처 중요도 시각화

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

ftr_importances_values = rf_clf1.feature_importances_
ftr_importances = pd.Series(ftr_importances_values, index=x_train.columns)
ftr_top_20 = ftr_importances_values.sort_values(ascending=False)[:20]

plt.figure(figsize=(8, 6))
sns.barplot(x=ftr_top_20, y=ftr_top_20.index)
plt.show()